<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 14


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Supplier в C#, который будет представлять информацию о
поставщиках товаров или услуг. На основе этого класса разработать 2-3
производных класса, демонстрирующих принципы наследования и полиморфизма.
В каждом из классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

namespace SupplierManagement
{
    // Делегаты
    public delegate void OrderProcessedHandler(string supplierName, string orderId, decimal amount);
    public delegate void SupplierRatingHandler(string supplierName, decimal oldRating, decimal newRating);

    // События
    public class SupplierEvents
    {
        public event OrderProcessedHandler OnOrderProcessed;
        public event SupplierRatingHandler OnRatingChanged;
        public event Action<string> OnSupplierStatusChanged;

        public void TriggerOrderProcessed(string supplier, string order, decimal amount) => 
            OnOrderProcessed?.Invoke(supplier, order, amount);
        
        public void TriggerRatingChange(string supplier, decimal oldRate, decimal newRate) => 
            OnRatingChanged?.Invoke(supplier, oldRate, newRate);
        
        public void TriggerStatusChange(string supplier) => 
            OnSupplierStatusChanged?.Invoke(supplier);
    }

    public interface ISupplierNotifier
    {
        void SendAlert(string message);
        bool CanSend { get; }
    }

    public interface IPaymentHandler
    {
        bool HandlePayment(decimal amount, string currency);
    }

    public abstract class SupplierBase : ISupplierNotifier
    {
        public int Id { get; }
        public string Name { get; }
        public string ContactPhone { get; set; }
        public string ContactEmail { get; set; }
        public decimal SuccessRate { get; private set; }
        public bool IsOperational { get; set; } = true;
        public DateTime RegistrationDate { get; }
        
        // Оптимизированные атрибуты
        public string LegalAddress { get; set; }
        public int ContractMonths { get; set; } = 12;
        public decimal ReliabilityScore { get; private set; } = 80; // В процентах
        public Queue<string> PendingTasks { get; } = new();
        public HashSet<string> SupportedRegions { get; } = new();

        protected SupplierEvents Events { get; }

        protected SupplierBase(int id, string name, string phone, string email, SupplierEvents events)
        {
            Id = id;
            Name = name ?? throw new ArgumentNullException(nameof(name));
            ContactPhone = phone;
            ContactEmail = email;
            RegistrationDate = DateTime.Now;
            Events = events;
        }

        public virtual void ShowInfo()
        {
            Console.WriteLine($"#{Id} {Name} (рейтинг: {SuccessRate:F0}, надежность: {ReliabilityScore}%)");
        }

        public abstract decimal ComputeDiscount(decimal orderAmount);

        public void HandleOrder(string orderDetails, decimal amount)
        {
            var discount = ComputeDiscount(amount);
            var finalAmount = amount * (1 - discount);
            
            Console.WriteLine($"{Name} → заказ: {orderDetails}");
            Console.WriteLine($"Сумма: {amount:C} → {finalAmount:C} (скидка {discount:P0})");
            
            Events.TriggerOrderProcessed(Name, orderDetails, amount);
            AddToOrderHistory(orderDetails, amount);
        }

        public void AdjustSuccessRate(decimal newRating)
        {
            var oldRating = SuccessRate;
            SuccessRate = (SuccessRate + newRating) / 2;
            UpdateReliabilityScore();
            
            Events.TriggerRatingChange(Name, oldRating, SuccessRate);
            Console.WriteLine($"📊 Рейтинг {Name}: {oldRating:F0} → {SuccessRate:F0}");        //Проверка нового стыкера
        }

        // Оптимизированные методы
        public void AddSupportedRegion(string region)
        {
            if (SupportedRegions.Add(region))
                Console.WriteLine($"📍 {Name} + регион: {region}");
        }

        public void AddTask(string task)
        {
            PendingTasks.Enqueue(task);
            Console.WriteLine($"📋 {Name} + задача: {task}");
        }

        public void ProcessNextTask()
        {
            if (PendingTasks.TryDequeue(out var task))
                Console.WriteLine($"✅ {Name} выполнил: {task}");
        }

        public void UpdateReliabilityScore()
        {
            ReliabilityScore = Math.Min(100, ReliabilityScore + 5);
        }

        protected virtual void AddToOrderHistory(string order, decimal amount)
        {
            // Базовая реализация
        }

        public virtual bool IsValid() => 
            !string.IsNullOrWhiteSpace(Name) &&
            !string.IsNullOrWhiteSpace(ContactEmail) &&
            !string.IsNullOrWhiteSpace(ContactPhone);

        bool ISupplierNotifier.CanSend => !string.IsNullOrWhiteSpace(ContactEmail) && IsOperational;

        void ISupplierNotifier.SendAlert(string message)
        {
            if (((ISupplierNotifier)this).CanSend)
                Console.WriteLine($"🔔 {Name}: {message}");
        }
    }

    public class GoodsSupplier : SupplierBase, IPaymentHandler
    {
        public string ProductCategory { get; }
        public int AvailableStock { get; private set; } = 100;
        public HashSet<string> ProductLines { get; } = new();
        public string StorageLocation { get; set; }
        public bool AcceptsLargeOrders { get; set; }
        public decimal DeliveryFee { get; set; } = 500;
        
        // Оптимизированные коллекции
        public Dictionary<string, int> ProductStock { get; } = new();
        public Queue<decimal> RecentSales { get; } = new();

        public GoodsSupplier(int id, string name, string phone, string email, string category, SupplierEvents events)
            : base(id, name, phone, email, events)
        {
            ProductCategory = category ?? "Разное";
        }

        public override void ShowInfo()
        {
            base.ShowInfo();
            Console.WriteLine($"   Категория: {ProductCategory}, Товаров: {ProductLines.Count}");
            if (!string.IsNullOrEmpty(StorageLocation))
                Console.WriteLine($"   Склад: {StorageLocation}");
        }

        public override decimal ComputeDiscount(decimal orderAmount)
        {
            var discount = 0m;
            
            // Круглые пороги скидок
            if (orderAmount > 20000) discount += 0.05m;      // 5%
            if (orderAmount > 50000) discount += 0.10m;      // 10%
            if (AcceptsLargeOrders && orderAmount > 100000) discount += 0.05m; // 5%
            
            // Дополнительная скидка за объем заказов
            if (RecentSales.Count > 5) discount += 0.02m;    // 2%
            
            return discount;
        }

        public void RegisterProduct(string productName, decimal price = 0)
        {
            if (ProductLines.Add(productName))
            {
                AvailableStock += 50; // Круглое число
                ProductStock[productName] = AvailableStock;
                Console.WriteLine($"✅ {Name} + товар: {productName} (запас: {AvailableStock})");
            }
        }

        public void RecordSale(decimal saleAmount)
        {
            if (RecentSales.Count >= 10) RecentSales.Dequeue();
            RecentSales.Enqueue(saleAmount);
        }

        public void ScheduleDelivery(int daysFromNow)
        {
            var date = DateTime.Now.AddDays(daysFromNow);
            Console.WriteLine($"📅 {Name} + поставка через {daysFromNow} дней");
        }

        public void UpdateProductPrice(string product, decimal newPrice)
        {
            Console.WriteLine($"💰 {Name} обновил цену {product}: {newPrice:C}");
        }

        public decimal CalculateAverageSale() => 
            RecentSales.Any() ? RecentSales.Average() : 0;

        public void ModifyStock(int quantityChange)
        {
            AvailableStock += quantityChange;
            Console.WriteLine($"📦 {Name}: запас {quantityChange:+0;-0;0} → {AvailableStock} шт.");
        }

        public bool CheckStock(string product, int requiredQty) => 
            ProductLines.Contains(product) && AvailableStock >= requiredQty;

        protected override void AddToOrderHistory(string order, decimal amount)
        {
            RecordSale(amount);
            AddTask($"Обработать заказ: {order}");
        }

        public bool HandlePayment(decimal amount, string currency)
        {
            var processingFee = amount * 0.02m; // Ровные 2%
            Console.WriteLine($"💰 {Name}: платеж {amount + processingFee:C} ({currency})");
            return true;
        }
    }

    public class ServiceProvider : SupplierBase, IPaymentHandler
    {
        public string ServiceCategory { get; }
        public decimal RatePerHour { get; set; } = 1000; // Круглая ставка
        public List<string> ServiceAreas { get; } = new();
        public int SpecialistsCount { get; set; } = 5; // Круглое число
        public bool ProvidesUrgentSupport { get; set; }
        
        // Оптимизированные коллекции
        public Dictionary<string, int> SpecialistSkills { get; } = new();
        public Queue<string> CompletedProjects { get; } = new();

        public ServiceProvider(int id, string name, string phone, string email, string serviceType, SupplierEvents events)
            : base(id, name, phone, email, events)
        {
            ServiceCategory = serviceType ?? "Общие услуги";
        }

        public override void ShowInfo()
        {
            base.ShowInfo();
            Console.WriteLine($"   Услуги: {ServiceCategory}, Ставка: {RatePerHour:C}/час");
            Console.WriteLine($"   Специалистов: {SpecialistsCount}");
        }

        public override decimal ComputeDiscount(decimal orderAmount)
        {
            // Простые пороги скидок
            return orderAmount > 20000 ? 0.15m : 0.10m; // 15% или 10%
        }

        public void AddServiceArea(string area)
        {
            ServiceAreas.Add(area);
            AddSupportedRegion(area);
            Console.WriteLine($"🌍 {Name} + регион: {area}");
        }

        public void AddSpecialist(string skill, int experienceYears = 1)
        {
            SpecialistSkills[skill] = experienceYears;
            SpecialistsCount = SpecialistSkills.Count;
            Console.WriteLine($"👨‍💻 {Name} + специалист: {skill} ({experienceYears} год)");
        }

        public void AddTechnology(string technology)
        {
            Console.WriteLine($"⚙️ {Name} освоил: {technology}");
        }

        public void ScheduleAppointment(int daysFromNow)
        {
            Console.WriteLine($"🕐 {Name} + встреча через {daysFromNow} дней");
        }

        public void CompleteProject(string projectName)
        {
            if (CompletedProjects.Count >= 5) CompletedProjects.Dequeue();
            CompletedProjects.Enqueue(projectName);
            
            AdjustSuccessRate(10); // Круглое число
            Console.WriteLine($"🎉 {Name} завершил: {projectName}");
        }

        public void ProcessNextAppointment()
        {
            Console.WriteLine($"✅ {Name} провел встречу");
        }

        public decimal QuoteProject(int estimatedHours)
        {
            var total = estimatedHours * RatePerHour;
            var discount = ComputeDiscount(total);
            return total * (1 - discount);
        }

        public bool CanProvideUrgentService() => ProvidesUrgentSupport;

        protected override void AddToOrderHistory(string order, decimal amount)
        {
            CompleteProject(order);
            ScheduleAppointment(1); // Завтра
        }

        public bool HandlePayment(decimal amount, string currency)
        {
            AddTask($"Подтвердить платеж {amount:C}");
            Console.WriteLine($"💳 {Name}: оплата услуг {amount:C} ({currency})");
            return true;
        }
    }

    public class EcoFriendlySupplier : GoodsSupplier
    {
        public bool HasEcoCertificate { get; set; }
        public decimal EnvironmentalImpact { get; private set; } = 10; // кг CO₂
        public bool UsesGreenEnergy { get; set; }
        public decimal WasteRecyclingRatio { get; set; } = 80; // В процентах
        
        // Оптимизированные коллекции
        public List<string> SustainablePractices { get; } = new();
        public Queue<string> EnvironmentalInitiatives { get; } = new();

        public EcoFriendlySupplier(int id, string name, string phone, string email, string category, SupplierEvents events)
            : base(id, name, phone, email, category, events) { }

        public override void ShowInfo()
        {
            base.ShowInfo();
            var certStatus = HasEcoCertificate ? "сертифицирован" : "без сертификата";
            Console.WriteLine($"   Экология: {certStatus}, CO₂: {EnvironmentalImpact}кг");
            Console.WriteLine($"   Переработка: {WasteRecyclingRatio}%");
        }

        public override decimal ComputeDiscount(decimal orderAmount)
        {
            var baseDiscount = base.ComputeDiscount(orderAmount);
            
            // Простые бонусы за экологичность
            if (HasEcoCertificate) baseDiscount += 0.05m;    // 5%
            if (UsesGreenEnergy) baseDiscount += 0.03m;      // 3%
            if (WasteRecyclingRatio > 80) baseDiscount += 0.02m; // 2%
            
            return baseDiscount;
        }

        public void AddSustainablePractice(string practice)
        {
            SustainablePractices.Add(practice);
            EnvironmentalImpact -= 2; // Уменьшаем на 2 кг
            Console.WriteLine($"🌱 {Name} + практика: {practice} (CO₂: {EnvironmentalImpact}кг)");
        }

        public void AddEcoCertification(string certification)
        {
            HasEcoCertificate = true;
            Console.WriteLine($"🏆 {Name} + сертификат: {certification}");
        }

        public void ProposeInitiative(string initiative)
        {
            EnvironmentalInitiatives.Enqueue(initiative);
            Console.WriteLine($"💡 {Name} + инициатива: {initiative}");
        }

        public void ProcessNextInitiative()
        {
            if (EnvironmentalInitiatives.TryDequeue(out var initiative))
            {
                AddSustainablePractice(initiative);
                Events.TriggerStatusChange(Name);
            }
        }

        public decimal CalculateSustainabilityBonus()
        {
            var bonus = (100 - EnvironmentalImpact) * 100; // Простая формула
            return bonus;
        }

        protected override void AddToOrderHistory(string order, decimal amount)
        {
            base.AddToOrderHistory(order, amount);
            RecordSale(amount);
            ProposeInitiative($"Оптимизация: {order}");
        }
    }

    public class SupplierCoordinator
    {
        private readonly List<SupplierBase> _suppliers;
        private readonly INotificationService _notifier;
        private readonly SupplierEvents _events;

        // Оптимизированные коллекции
        public Dictionary<int, SupplierBase> SupplierDictionary { get; } = new();
        public HashSet<string> AllSupportedRegions { get; } = new();

        public SupplierCoordinator(INotificationService notifier, SupplierEvents events)
        {
            _suppliers = new List<SupplierBase>();
            _notifier = notifier;
            _events = events;
            
            SetupEventHandlers();
        }

        private void SetupEventHandlers()
        {
            _events.OnOrderProcessed += (supplier, order, amount) =>
            {
                Console.WriteLine($"📦 [Событие] {supplier} → {order} ({amount:C})");
            };

            _events.OnRatingChanged += (supplier, oldRate, newRate) =>
            {
                Console.WriteLine($"⭐ [Событие] {supplier}: {oldRate:F0} → {newRate:F0}");
            };
        }

        public void RegisterSupplier(SupplierBase supplier)
        {
            if (supplier.IsValid())
            {
                _suppliers.Add(supplier);
                SupplierDictionary[supplier.Id] = supplier;
                _notifier.SendRegistrationAlert(supplier.Name, supplier.ContactEmail);
            }
        }

        public SupplierBase FindById(int id) => SupplierDictionary.GetValueOrDefault(id);

        public IEnumerable<SupplierBase> GetActiveSuppliers() => _suppliers.Where(s => s.IsOperational);

        public void ExecutePayments(decimal amount, string currency)
        {
            var paymentHandlers = _suppliers.OfType<IPaymentHandler>().ToList();
            paymentHandlers.ForEach(handler => handler.HandlePayment(amount, currency));
        }

        // Оптимизированные методы
        public void DisplaySupplierStatistics()
        {
            Console.WriteLine($"\n--- СТАТИСТИКА ---");
            Console.WriteLine($"Поставщиков: {_suppliers.Count}");
            Console.WriteLine($"Регионов: {AllSupportedRegions.Count}");
            
            var avgRating = _suppliers.Average(s => s.SuccessRate);
            Console.WriteLine($"Средний рейтинг: {avgRating:F0}");
        }

        public void ProcessAllPendingTasks()
        {
            Console.WriteLine($"\n--- ЗАДАЧИ ---");
            foreach (var supplier in _suppliers)
            {
                supplier.ProcessNextTask();
            }
        }
    }

    public interface INotificationService
    {
        void SendRegistrationAlert(string supplierName, string email);
    }

    public class SupplierNotificationService : INotificationService
    {
        public void SendRegistrationAlert(string supplierName, string email)
        {
            Console.WriteLine($"✉️  Зарегистрирован: {supplierName}");
        }
    }

    class Program
    {
        static void Main()
        {
            var events = new SupplierEvents();
            var notifier = new SupplierNotificationService();
            var coordinator = new SupplierCoordinator(notifier, events);

            var supplierList = new SupplierBase[]
            {
                new SupplierBase(101, "Поставщик-ОСНОВА", "+79990001122", "main@example.com", events),
                new GoodsSupplier(102, "ТехноПоставки", "+79993334455", "tech@example.com", "Электроника", events),
                new ServiceProvider(103, "ПрофУслуги", "+79996667788", "services@example.com", "IT-аутсорсинг", events),
                new EcoFriendlySupplier(104, "ЭкоПродукт", "+79991234567", "eco@example.com", "Органика", events)
            };

            Array.ForEach(supplierList, coordinator.RegisterSupplier);

            // Настройка поставщиков
            var techSupplier = (GoodsSupplier)supplierList[1];
            techSupplier.RegisterProduct("Ноутбуки", 50000);
            techSupplier.RegisterProduct("Мониторы", 20000);
            techSupplier.StorageLocation = "Склад-1";
            techSupplier.AcceptsLargeOrders = true;
            techSupplier.AddSupportedRegion("Москва");
            techSupplier.ScheduleDelivery(7);

            var serviceProvider = (ServiceProvider)supplierList[2];
            serviceProvider.RatePerHour = 1500;
            serviceProvider.AddServiceArea("Центр");
            serviceProvider.AddSpecialist("C#", 5);
            serviceProvider.AddTechnology("Docker");

            var ecoSupplier = (EcoFriendlySupplier)supplierList[3];
            ecoSupplier.AddEcoCertification("Органик");
            ecoSupplier.UsesGreenEnergy = true;
            ecoSupplier.AddSustainablePractice("Солнечные панели");
            ecoSupplier.ProposeInitiative("Эко-упаковка");

            Console.WriteLine("\n" + new string('=', 40));
            Console.WriteLine("ПОСТАВЩИКИ");
            Console.WriteLine(new string('=', 40));
            
            Array.ForEach(supplierList, s => { s.ShowInfo(); Console.WriteLine(); });

            Console.WriteLine("ОБРАБОТКА ЗАКАЗОВ");
            Console.WriteLine(new string('-', 25));
            
            var testOrders = new decimal[] { 10000, 30000, 60000, 40000 };
            for (int i = 0; i < supplierList.Length; i++)
            {
                supplierList[i].HandleOrder($"Заказ-{i + 1}", testOrders[i]);
            }

            // Дополнительные операции
            techSupplier.RecordSale(25000);
            techSupplier.RecordSale(30000);
            Console.WriteLine($"\nСредние продажи: {techSupplier.CalculateAverageSale():C}");

            serviceProvider.CompleteProject("Внедрение CRM");
            ecoSupplier.ProcessNextInitiative();

            Console.WriteLine("\nПЛАТЕЖИ");
            Console.WriteLine(new string('-', 25));
            coordinator.ExecutePayments(20000, "RUB");

            coordinator.DisplaySupplierStatistics();
            coordinator.ProcessAllPendingTasks();

            Console.WriteLine("\nУВЕДОМЛЕНИЯ");
            Console.WriteLine(new string('-', 25));
            foreach (var supplier in supplierList)
            {
                ((ISupplierNotifier)supplier).SendAlert("Проверка системы");
            }
        }
    }
}

In [ ]:
✉️  Зарегистрирован: Поставщик-ОСНОВА
✉️  Зарегистрирован: ТехноПоставки
✉️  Зарегистрирован: ПрофУслуги
✉️  Зарегистрирован: ЭкоПродукт
✅ ТехноПоставки + товар: Ноутбуки (запас: 150)
✅ ТехноПоставки + товар: Мониторы (запас: 200)
📅 ТехноПоставки + поставка через 7 дней
🌍 ПрофУслуги + регион: Центр
👨‍💻 ПрофУслуги + специалист: C# (5 год)
⚙️ ПрофУслуги освоил: Docker
🏆 ЭкоПродукт + сертификат: Органик
🌱 ЭкоПродукт + практика: Солнечные панели (CO₂: 8кг)
💡 ЭкоПродукт + инициатива: Эко-упаковка

========================================
ПОСТАВЩИКИ
========================================
#101 Поставщик-ОСНОВА (рейтинг: 0, надежность: 80%)

#102 ТехноПоставки (рейтинг: 0, надежность: 80%)
   Категория: Электроника, Товаров: 2
   Склад: Склад-1

#103 ПрофУслуги (рейтинг: 0, надежность: 80%)
   Услуги: IT-аутсорсинг, Ставка: ₽1,500.00/час
   Специалистов: 1

#104 ЭкоПродукт (рейтинг: 0, надежность: 80%)
   Категория: Органика, Товаров: 0
   Экология: сертифицирован, CO₂: 8кг
   Переработка: 80%

ОБРАБОТКА ЗАКАЗОВ
-------------------------
Поставщик-ОСНОВА → заказ: Заказ-1
Сумма: ₽10,000.00 → ₽10,000.00 (скидка 0%)
📦 [Событие] Поставщик-ОСНОВА → Заказ-1 (₽10,000.00)

ТехноПоставки → заказ: Заказ-2
Сумма: ₽30,000.00 → ₽28,500.00 (скидка 5%)
📦 [Событие] ТехноПоставки → Заказ-2 (₽30,000.00)

ТехноПоставки → заказ: Заказ-3
Сумма: ₽60,000.00 → ₽51,000.00 (скидка 15%)
📦 [Событие] ТехноПоставки → Заказ-3 (₽60,000.00)

ПрофУслуги → заказ: Заказ-4
Сумма: ₽40,000.00 → ₽34,000.00 (скидка 15%)
📦 [Событие] ПрофУслуги → Заказ-4 (₽40,000.00)
🎉 ПрофУслуги завершил: Заказ-4
📊 Рейтинг ПрофУслуги: 0 → 5
⭐ [Событие] ПрофУслуги: 0 → 5
🕐 ПрофУслуги + встреча через 1 дней

Средние продажи: ₽27,500.00
🎉 ПрофУслуги завершил: Внедрение CRM
📊 Рейтинг ПрофУслуги: 5 → 8
⭐ [Событие] ПрофУслуги: 5 → 8
🌱 ЭкоПродукт + практика: Эко-упаковка (CO₂: 6кг)
🔄 [Событие] Статус ЭкоПродукт обновлен

ПЛАТЕЖИ
-------------------------
💰 ТехноПоставки: платеж ₽20,400.00 (RUB)
💳 ПрофУслуги: оплата услуг ₽20,000.00 (RUB)
💰 ЭкоПродукт: платеж ₽20,400.00 (RUB)

--- СТАТИСТИКА ---
Поставщиков: 4
Регионов: 0
Средний рейтинг: 3

--- ЗАДАЧИ ---
✅ Поставщик-ОСНОВА выполнил: Обработать заказ: Заказ-1
✅ ТехноПоставки выполнил: Обработать заказ: Заказ-2
✅ ПрофУслуги выполнил: Подтвердить платеж ₽40,000.00
✅ ЭкоПродукт выполнил: Обработать заказ: Заказ-4

УВЕДОМЛЕНИЯ
-------------------------
🔔 Поставщик-ОСНОВА: Проверка системы
🔔 ТехноПоставки: Проверка системы
🔔 ПрофУслуги: Проверка системы
🔔 ЭкоПродукт: Проверка системы